Trabajo Final de Depósito y Mineria de Datos - Cursada 2024  
 Integrantes:  
    Francisco Repetto FAI-2548   
    Malena Rivera     FAI-2516

# Limpieza de Archivos Data Lake

In [1]:
import time
from pyspark.sql import SparkSession

t0 = time.time()
spark = (
    SparkSession.builder
    .appName("DataLake-ELT")
    .master("local[*]")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


## Limpieza de CSV (Datos Estructurados)

#### Leemos el csv sucio que deseamos limpiar

In [2]:
df_credito = (
    spark.read
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .csv("/data/raw/csv_sucio/credit_card_fraud_10k_dirty.csv")
)
df_credito.printSchema()

# Vemos la estructura que detecto spark
df_credito.count()

root
 |-- transaction_id: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- transaction_hour: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- foreign_transaction: string (nullable = true)
 |-- location_mismatch: string (nullable = true)
 |-- device_trust_score: string (nullable = true)
 |-- velocity_last_24h: string (nullable = true)
 |-- cardholder_age: string (nullable = true)
 |-- is_fraud: string (nullable = true)



10300

In [3]:
df_credito.limit(5).show(truncate=False)

+------------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+
|transaction_id    |amount|transaction_hour|merchant_category|foreign_transaction|location_mismatch|device_trust_score|velocity_last_24h|cardholder_age|is_fraud|
+------------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+
|1.0               |84.47 |22.0            |Electronics      |0.0                |0.0              |66.0              |3.0              |40.0          |0.0     |
|2.0               |541.82|3.0             |Travel           |1.0                |0.0              |87.0              |1.0              |64.0          |0.0     |
|3.0               |237.01|17.0            |Grocery          |0.0                |0.0              |49.0              |1.0              |61.0          |0.0     |
|249876.49791231734|164.33|4

Como podemos observar, el csv cuenta con 10300 filas y cuenta con 10 columnas las cuales son:  
1. transaccion_id: identificador unico  
2. cantidad: monto de la transaccion  
3. hora_de_transaccion: hora de la transaccion (0-23)  
4. categoría_comercial: tipo de comerciante  
5. transaccion_extranjera: Indica si la transaccion es internacional (0/1)  
6. discrepancia_de_ubicacion: desajuste entre facturacion y ubicacion de transaccion (0/1)
7. confianza_del_dispositivo_usado: puntuacion de confianza del dispositivo (0-100)
8. velocidad_ultimas_24h: Numero de transacciones en las ultimas 24hs
9. edad_del_titular: edad del titular de la tarjeta
10. es_fraude: Variable objetivo. Indica si es fraude o no (0=normal | 1=fraude)

Por otro lado, tambien podemos observar que muchas de las filas deben ser numericas tal como se describe anteriormente. Sin embargo, todas las columnas son del tipo str, es decir, que se guardo texto y no numeros. 

Procederemos a la limpieza del mismo

### ETAPA 1 - COERCION DE TIPOS  

Vamos a cambiar el tipo de todas aquellas columnas que deben ser numericas y actualmente son del tipo str.
En el caso de que haya algun error durante la convercion se dejará en blanco esa celda (errors=coerce)

In [4]:
#etapa 1- coercion (forzamos todo a numero)
from pyspark.sql import functions as F

cols_numericas = [
    'transaction_id', 
    'amount', 
    'transaction_hour', 
    'foreign_transaction', 
    'location_mismatch', 
    'device_trust_score', 
    'velocity_last_24h', 
    'cardholder_age', 
    'is_fraud'
]

for c in cols_numericas:
    df_credito = df_credito.withColumn(c, F.col(c).cast("double"))

In [5]:
#comprobamos el tipo ahora
df_credito.printSchema()

root
 |-- transaction_id: double (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_hour: double (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- foreign_transaction: double (nullable = true)
 |-- location_mismatch: double (nullable = true)
 |-- device_trust_score: double (nullable = true)
 |-- velocity_last_24h: double (nullable = true)
 |-- cardholder_age: double (nullable = true)
 |-- is_fraud: double (nullable = true)



Podemos ver ahora que las columnas numericas son del tipo double.
Eso puede NO ser del todo correcto ya que hay algunas columnas que no deben tener decimales como por ejemplo 'cardholder_age' que guarda la edad del titular o 'is_fraud' que es una variable booleana y solo guarda valores 0/1

In [6]:
#veamos ahora la cantidad de nulos por columna
df_credito.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_credito.columns
]).show(truncate=False)

+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+
|transaction_id|amount|transaction_hour|merchant_category|foreign_transaction|location_mismatch|device_trust_score|velocity_last_24h|cardholder_age|is_fraud|
+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+
|698           |694   |704             |504              |504                |486              |518               |513              |498           |501     |
+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+



Empezemos por la primer columna

In [7]:
print("antes")
df_credito.select("transaction_id").distinct().limit(10).show(truncate=False)

antes
+------------------+
|transaction_id    |
+------------------+
|299.0             |
|305.0             |
|249876.49791231734|
|496.0             |
|558.0             |
|596.0             |
|692.0             |
|769.0             |
|934.0             |
|1051.0            |
+------------------+



El id deberia ser incremental y del tipo entero, por lo tanto, vamos a cambiar todas las filas que sean raras para que sigan el orden preestablecido

In [8]:
from pyspark.sql.window import Window

# ordenar
df_credito = df_credito.orderBy("transaction_id")

# eliminar ids inválidos
df_credito = df_credito.filter(
    F.col("transaction_id") == F.floor(F.col("transaction_id"))
)

# reconstruir secuencia
window = Window.orderBy("transaction_id")
df_credito = df_credito.withColumn(
    "transaction_id",
    F.row_number().over(window)
)

In [9]:
print("Despues")
df_credito.select("transaction_id").distinct().limit(10).show(truncate=False)

Despues
+--------------+
|transaction_id|
+--------------+
|1             |
|2             |
|3             |
|4             |
|5             |
|6             |
|7             |
|8             |
|9             |
|10            |
+--------------+



## ETAPA 2 - Tratamiento de Nulos

In [10]:
#vemos cuantos valores faltantes tiene la columna 'is_fraud'
df_credito.groupBy("is_fraud") \
    .count() \
    .orderBy("is_fraud") \
    .show()


+------------------+-----+
|          is_fraud|count|
+------------------+-----+
|              NULL|  465|
|               0.0| 8788|
|0.7656186198448346|   92|
|               1.0|  138|
+------------------+-----+



Podemos observar que no solo hay valores nulos (Nan) sino valores extraños  
Recordemos que esta columna solo debe guardar valores 0 y 1 donde 0 indica que NO hubo fraude y 1 que sí

Vamos a llenar con 0 (dado que es el valor mas comun) a las filas vacias  
Aquellas que tienen un valor extraño las borraremos dado que son pocas

In [11]:
# Vamos a llenar las celdas en blanco con 0, es decir, que no hubo fraude
df_credito = df_credito.fillna({"is_fraud": 0})

antes_nulos = df_credito.count()

# Nos quedamos SOLO con las filas donde el valor es exactamente 0 o 1
# Esto elimina cualquier "valor extraño" (como 2, 99, o strings raros)
df_credito = df_credito.filter(F.col("is_fraud").isin(0, 1))

df_credito = df_credito.withColumn(
    "is_fraud",
    F.col("is_fraud").cast("int")
)


despues = df_credito.count()

print(f"Filas eliminadas en is_fraud: {antes_nulos - despues}")


df_credito.groupBy("is_fraud") \
    .count() \
    .orderBy("is_fraud") \
    .show()


Filas eliminadas en is_fraud: 92
+--------+-----+
|is_fraud|count|
+--------+-----+
|       0| 9253|
|       1|  138|
+--------+-----+



Como podemos ver hay más cantidad de 0 que de 1, eso indica que va a estar más influenciado de que no hay fraude de que sí

A continuacion, aplicaremos la misma logica con el resto de variable booleanas

In [12]:
#Empezamos con 'foreing_transaction' que indica si la transaccion fue extranjera (1) o no (0)
#Veamos la cantidad de nulos y si tiene valores raros
df_credito.groupBy("foreign_transaction") \
          .count() \
          .orderBy(F.desc("count")) \
          .show()

+-------------------+-----+
|foreign_transaction|count|
+-------------------+-----+
|                0.0| 7958|
|                1.0|  877|
|               NULL|  463|
|  4.923391215526047|   93|
+-------------------+-----+



In [13]:
from pyspark.sql.functions import col
# Solo debe tener 0 y 1
df_credito = df_credito.fillna({"foreign_transaction": 0})

# Cantidad de filas antes
antes_nulos = df_credito.count()

# Nos quedamos SOLO con 0 o 1
df_credito = df_credito.filter(col("foreign_transaction").isin([0, 1]))

# Cast a int
df_credito = df_credito.withColumn(
    "foreign_transaction",
    col("foreign_transaction").cast("int")
)

# Cantidad de filas después
despues = df_credito.count()

print(f"Filas eliminadas en foreign_transaction: {antes_nulos - despues}")


Filas eliminadas en foreign_transaction: 93


In [14]:
df_credito.groupBy("foreign_transaction") \
          .count() \
          .orderBy(F.desc("count")) \
          .show()

+-------------------+-----+
|foreign_transaction|count|
+-------------------+-----+
|                  0| 8421|
|                  1|  877|
+-------------------+-----+



In [15]:
#Nuevamente la misma logica ahora con location_mismatch que indica si hay discrepancias entre la facturacion y la compra
#Veamos la cantidad de Nulos y si tiene valores extraños
df_credito.groupBy("location_mismatch") \
          .count() \
          .orderBy(F.desc("count")) \
          .show()

+-----------------+-----+
|location_mismatch|count|
+-----------------+-----+
|              0.0| 8007|
|              1.0|  747|
|             NULL|  443|
|4.309730005094243|  101|
+-----------------+-----+



In [16]:
# Solo debe tener 0 y 1
df_credito = df_credito.fillna({"location_mismatch": 0})

# Cantidad de filas antes
antes_nulos = df_credito.count()

# Nos quedamos SOLO con 0 o 1
df_credito = df_credito.filter(col("location_mismatch").isin([0, 1]))

# Cast a int
df_credito = df_credito.withColumn(
    "location_mismatch",
    col("location_mismatch").cast("int")
)

# Cantidad de filas después
despues = df_credito.count()

print(f"Filas eliminadas en location_mismatch: {antes_nulos - despues}")


Filas eliminadas en location_mismatch: 101


In [17]:
df_credito.groupBy("location_mismatch") \
          .count() \
          .orderBy(F.desc("count")) \
          .show()

+-----------------+-----+
|location_mismatch|count|
+-----------------+-----+
|                0| 8450|
|                1|  747|
+-----------------+-----+



Para las variables continuas tambien debemos tratar sus nulos y valores extraños  
En estos casos rellenaremos los nulos con la mediana


In [18]:
from pyspark.sql.functions import col, sum as spark_sum, when

cols_rellenar_mediana = [
    'amount',
    'cardholder_age',
    'device_trust_score',
    'velocity_last_24h'
]

for c in cols_rellenar_mediana:
    nulos = df_credito.select(
        spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias("nulos")
    ).collect()[0]["nulos"]

    print(f"{c} tiene: {nulos} nulos")


amount tiene: 624 nulos
cardholder_age tiene: 450 nulos
device_trust_score tiene: 469 nulos
velocity_last_24h tiene: 456 nulos


In [19]:
cols_rellenar_mediana = [
    'amount',
    'cardholder_age',
    'device_trust_score',
    'velocity_last_24h'
]

for c in cols_rellenar_mediana:
    # 1. calcular mediana (percentil 50)
    mediana = df_credito.approxQuantile(c, [0.5], 0.0)[0]

    # 2. rellenar nulos con la mediana
    df_credito = df_credito.withColumn(
        c,
        when(col(c).isNull(), mediana).otherwise(col(c))
    )

    # 3. verificar nulos
    nulos = df_credito.select(
        spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias("nulos")
    ).collect()[0]["nulos"]

    print(f"{c} tiene: {nulos} nulos")


amount tiene: 0 nulos
cardholder_age tiene: 0 nulos
device_trust_score tiene: 0 nulos
velocity_last_24h tiene: 0 nulos


In [20]:
cols_rellenar_mediana = [
    'amount',
    'cardholder_age',
    'device_trust_score',
    'velocity_last_24h'
]

for c in cols_rellenar_mediana:

    # Cuartiles
    Q1 = df_credito.approxQuantile(c, [0.25], 0.0)[0]
    Q3 = df_credito.approxQuantile(c, [0.75], 0.0)[0]
    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    # Outliers
    outliers = df_credito.filter(
        (col(c) < limite_inferior) | (col(c) > limite_superior)
    )

    print(f"Columna: {c}")
    print(f"Cantidad de outliers detectados: {outliers.count()}")
    print(f"Límite superior: {limite_superior}")

    # Primeros 10 valores fuera de rango
    outliers.select(c).show(10, truncate=False)

    # Comparaciones
    mediana = df_credito.approxQuantile(c, [0.5], 0.0)[0]
    maximo = df_credito.agg({c: "max"}).collect()[0][0]

    print(f"Mediana: {mediana}")
    print(f"Valor máximo: {maximo}")
    print("-" * 30)


Columna: amount
Cantidad de outliers detectados: 597
Límite superior: 504.41499999999996
+------+
|amount|
+------+
|541.82|
|630.64|
|535.26|
|535.26|
|628.71|
|504.92|
|780.15|
|515.36|
|642.2 |
|590.76|
+------+
only showing top 10 rows

Mediana: 123.88
Valor máximo: nan
------------------------------
Columna: cardholder_age
Cantidad de outliers detectados: 89
Límite superior: 93.5
+------------------+
|cardholder_age    |
+------------------+
|2168.5517593064765|
|2168.5517593064765|
|2168.5517593064765|
|2168.5517593064765|
|2168.5517593064765|
|2168.5517593064765|
|2168.5517593064765|
|2168.5517593064765|
|2168.5517593064765|
|2168.5517593064765|
+------------------+
only showing top 10 rows

Mediana: 44.0
Valor máximo: 2168.5517593064765
------------------------------
Columna: device_trust_score
Cantidad de outliers detectados: 97
Límite superior: 132.5
+------------------+
|device_trust_score|
+------------------+
|3091.5985282093216|
|3091.5985282093216|
|3091.5985282093216|
|

In [21]:
df_limpio = df_credito

cols_a_limpiar = [
    'cardholder_age',
    'device_trust_score',
    'velocity_last_24h',
    'amount'
]

for c in cols_a_limpiar:

    Q1 = df_limpio.approxQuantile(c, [0.25], 0.0)[0]
    Q3 = df_limpio.approxQuantile(c, [0.75], 0.0)[0]
    IQR = Q3 - Q1

    factor = 3.0 if c == 'amount' else 1.5

    limite_inferior = Q1 - factor * IQR
    limite_superior = Q3 + factor * IQR

    antes = df_limpio.count()

    df_limpio = df_limpio.filter(
        (col(c) >= limite_inferior) &
        (col(c) <= limite_superior)
    )

    despues = df_limpio.count()

    print(f"Columna {c}: Se eliminaron {antes - despues} filas.")


Columna cardholder_age: Se eliminaron 89 filas.
Columna device_trust_score: Se eliminaron 95 filas.
Columna velocity_last_24h: Se eliminaron 135 filas.
Columna amount: Se eliminaron 192 filas.


In [22]:
print(f"\nTotal de filas eliminadas: {df_credito.count() - df_limpio.count()}")
print(f"Tamaño actual del dataset: {df_limpio.count()} filas.")



Total de filas eliminadas: 511
Tamaño actual del dataset: 8686 filas.


In [23]:
#Como se trabajo sobre una copia ahora hacemos que nuestro dataset sea la copia
df_credito = df_limpio

In [24]:
for c in cols_rellenar_mediana:

    # Cuartiles
    Q1 = df_limpio.approxQuantile(c, [0.25], 0.0)[0]
    Q3 = df_limpio.approxQuantile(c, [0.75], 0.0)[0]
    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    # Filtramos outliers
    outliers = df_limpio.filter(
        (col(c) < limite_inferior) | (col(c) > limite_superior)
    )

    print(f"Columna: {c}")
    print(f"Cantidad de outliers detectados: {outliers.count()}")
    print(f"Límite superior: {limite_superior}")

    # Primeros 5 valores "locos"
    outliers.select(c).show(5, truncate=False)

    # Mediana y máximo
    mediana = df_limpio.approxQuantile(c, [0.5], 0.0)[0]
    maximo = df_limpio.agg({c: "max"}).collect()[0][0]

    print(f"Mediana: {mediana}")
    print(f"Valor máximo: {maximo}")
    print("-" * 30)


Columna: amount
Cantidad de outliers detectados: 477
Límite superior: 480.135
+------+
|amount|
+------+
|541.82|
|630.64|
|628.71|
|504.92|
|515.36|
+------+
only showing top 5 rows

Mediana: 123.88
Valor máximo: 775.12
------------------------------
Columna: cardholder_age
Cantidad de outliers detectados: 0
Límite superior: 93.5
+--------------+
|cardholder_age|
+--------------+
+--------------+

Mediana: 44.0
Valor máximo: 69.0
------------------------------
Columna: device_trust_score
Cantidad de outliers detectados: 0
Límite superior: 131.5
+------------------+
|device_trust_score|
+------------------+
+------------------+

Mediana: 62.0
Valor máximo: 99.0
------------------------------
Columna: velocity_last_24h
Cantidad de outliers detectados: 0
Límite superior: 6.0
+-----------------+
|velocity_last_24h|
+-----------------+
+-----------------+

Mediana: 2.0
Valor máximo: 6.0
------------------------------


Como podemos notar, antes de la limpieza de valores tipicos habia bastantes inconsistencias. Por ejemplo
1. Habia transaccion con cantidades de 8813.402647488014
2. Habia titulares de la tarjeta con edades de 2168.5517593064765 años
3. Habia valores en la columna de las horas del dia (0-23) con valores de 100.6646216768916

Todo eso quedo erradicado con la limpieza hecha anteriormente. Podemos notar que ahora, por ejemplo, la cantidad más alta de una transaccion es de 773.35, la edad del titular mas grande es de 69 años pero se considera atipico si supera los 93.5 años años

In [25]:
#Vamos ahora con la columna de categoria de la compra. Esta columna tiene todos sus valores con texto
#Veamos si tiene nulos o valores extraños
df_credito.groupBy("merchant_category") \
          .count() \
          .orderBy(F.desc("count")) \
          .show()

+-----------------+-----+
|merchant_category|count|
+-----------------+-----+
|             Food| 1707|
|         Clothing| 1657|
|           Travel| 1598|
|      Electronics| 1564|
|          Grocery| 1561|
|             NULL|  427|
|             FOOD|   38|
|      ELECTRONICS|   37|
|          GROCERY|   30|
|         CLOTHING|   30|
|           TRAVEL|   27|
|              NAN|   10|
+-----------------+-----+



Podemos ver que hay filas que tienen categoria nan y otras NAN
- nan es el nulo de python es decir que no tienen nada
- NAN sin embargo, es texto, auqnue se quiere dar a entender q son valores vacios, por lo tanto lo remplazaremos

Tambien hay dos valores para la misma categoria, nada mas que uno está en mayusculas y otras en minusculas. Tambien limpiaremos 

In [26]:
from pyspark.sql.functions import  when, initcap
# Convertimos el string 'NAN' a nulo real
df_credito = df_credito.withColumn(
    "merchant_category",
    when(col("merchant_category") == "NAN", None)
    .otherwise(col("merchant_category"))
)
# Pasamos todo a minúsculas y capitalizamos
df_credito = df_credito.withColumn(
    "merchant_category",
    initcap(col("merchant_category"))
)
df_credito.groupBy("merchant_category").count().orderBy("count", ascending=False).show(truncate=False)

+-----------------+-----+
|merchant_category|count|
+-----------------+-----+
|Food             |1745 |
|Clothing         |1687 |
|Travel           |1625 |
|Electronics      |1601 |
|Grocery          |1591 |
|NULL             |437  |
+-----------------+-----+



In [27]:
from pyspark.sql.functions import desc
#Calcular la moda de merchant_category
moda_mercancia = (
    df_credito
    .groupBy("merchant_category")
    .count()
    .orderBy(desc("count"))
    .first()["merchant_category"]
)
#Rellenar los nulos con la moda
df_credito = df_credito.fillna(
    {"merchant_category": moda_mercancia}
)

#Ver conteo de valores (incluyendo los que eran nulos)
df_credito.groupBy("merchant_category").count().orderBy(desc("count")).show(truncate=False)


+-----------------+-----+
|merchant_category|count|
+-----------------+-----+
|Food             |2182 |
|Clothing         |1687 |
|Travel           |1625 |
|Electronics      |1601 |
|Grocery          |1591 |
+-----------------+-----+



In [28]:
#vamos ahora con la ultima columna: HORA de la transaccion
#Como siempre, veamos que valores tiene
df_credito.groupBy("transaction_hour") \
          .count() \
          .orderBy(F.desc("count")) \
          .show()

+----------------+-----+
|transaction_hour|count|
+----------------+-----+
|            NULL|  583|
|            14.0|  365|
|             7.0|  356|
|            23.0|  351|
|            13.0|  348|
|             8.0|  347|
|            18.0|  347|
|             9.0|  347|
|             1.0|  346|
|            15.0|  345|
|            22.0|  340|
|             5.0|  340|
|             3.0|  339|
|            17.0|  338|
|             0.0|  336|
|            21.0|  333|
|            19.0|  333|
|            20.0|  332|
|            16.0|  332|
|            12.0|  328|
+----------------+-----+
only showing top 20 rows



In [29]:
#hay 593 nulos asi q vamos a reemplazar eso por la moda
moda_hora = (
    df_credito
    .filter(col("transaction_hour").isNotNull())  
    .groupBy("transaction_hour")
    .count()
    .orderBy(desc("count"))
    .first()["transaction_hour"]
)

df_credito = df_credito.fillna(
    {"transaction_hour": moda_hora}
)
df_credito.groupBy("transaction_hour").count().orderBy(desc("count")).show()

+----------------+-----+
|transaction_hour|count|
+----------------+-----+
|            14.0|  956|
|             7.0|  356|
|            23.0|  351|
|            13.0|  348|
|             8.0|  347|
|            18.0|  347|
|             9.0|  347|
|             1.0|  346|
|            15.0|  345|
|            22.0|  340|
|             5.0|  340|
|             3.0|  339|
|            17.0|  338|
|             0.0|  336|
|            21.0|  333|
|            19.0|  333|
|            20.0|  332|
|            16.0|  332|
|            12.0|  328|
|             4.0|  315|
+----------------+-----+
only showing top 20 rows



In [30]:
#podemos ver que hay un valor atipico de hora 578, lo vamos a borrar
# Filtramos para quedarnos solo con lo que NO sea 578.740609 
# Filtramos SIN comillas porque es un número
# Usamos un margen pequeño por si hay problemas de redondeo en floats
# Borramos cualquier valor que sea mayor a 24 (porque una hora no puede ser 578!)

# Contamos filas antes
antes = df_credito.count()

# Filtramos horas inválidas (> 24)
df_credito = df_credito.filter(col("transaction_hour") <= 24)

# Contamos filas después
despues = df_credito.count()

print(f"Se eliminaron {antes - despues} filas.")

# Casteamos a entero
df_credito = df_credito.withColumn(
    "transaction_hour",
    col("transaction_hour").cast("int")
)

# Vemos la distribución
df_credito.groupBy("transaction_hour").count().orderBy("transaction_hour").show()

Se eliminaron 76 filas.
+----------------+-----+
|transaction_hour|count|
+----------------+-----+
|               0|  336|
|               1|  346|
|               2|  304|
|               3|  339|
|               4|  315|
|               5|  340|
|               6|  314|
|               7|  356|
|               8|  347|
|               9|  347|
|              10|  283|
|              11|  300|
|              12|  328|
|              13|  348|
|              14|  956|
|              15|  345|
|              16|  332|
|              17|  338|
|              18|  347|
|              19|  333|
+----------------+-----+
only showing top 20 rows



In [31]:
#ETAPA 3: CONVERSIÓN DE TIPOS (Formato Final)

In [32]:
# Ahora que no hay nulos ni basura, convertimos a enteros limpios, algunos ya están pero por las dudas
cols_a_entero = [
    'transaction_id', 'transaction_hour', 'cardholder_age',
    'is_fraud', 'foreign_transaction', 'location_mismatch'
]

for c in cols_a_entero:
    df_credito = df_credito.withColumn(
        c,
        col(c).cast("int")
    )

In [33]:
df_credito.printSchema()

root
 |-- transaction_id: integer (nullable = false)
 |-- amount: double (nullable = true)
 |-- transaction_hour: integer (nullable = true)
 |-- merchant_category: string (nullable = false)
 |-- foreign_transaction: integer (nullable = true)
 |-- location_mismatch: integer (nullable = true)
 |-- device_trust_score: double (nullable = true)
 |-- velocity_last_24h: double (nullable = true)
 |-- cardholder_age: integer (nullable = true)
 |-- is_fraud: integer (nullable = true)



In [34]:
antes = df_credito.count()

df_credito = df_credito.dropDuplicates()

despues = df_credito.count()

print(f"Se eliminaron {antes - despues} filas.")

Se eliminaron 0 filas.


Cambio columna categorica a numerica

In [35]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(
    inputCol="merchant_category",
    outputCol="merchant_category_index",
    handleInvalid="keep"
)

df_credito  = indexer.fit(df_credito).transform(df_credito)

# Eliminar la original
df_credito  = df_credito.drop("merchant_category")

# Opcional: renombrar
df_credito = df_credito.withColumnRenamed(
    "merchant_category_index",
    "merchant_category"
)

df_credito.printSchema()

root
 |-- transaction_id: integer (nullable = false)
 |-- amount: double (nullable = true)
 |-- transaction_hour: integer (nullable = true)
 |-- foreign_transaction: integer (nullable = true)
 |-- location_mismatch: integer (nullable = true)
 |-- device_trust_score: double (nullable = true)
 |-- velocity_last_24h: double (nullable = true)
 |-- cardholder_age: integer (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- merchant_category: double (nullable = false)



## Limpieza de Jsons (Datos semi-estructurados)

### Carga inicial de los archivos json

In [36]:
#Vemos que archivos hay en la carpeta /data/raw/jsons
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
    spark._jsc.hadoopConfiguration()
)

# Path a la carpeta
path = spark._jvm.org.apache.hadoop.fs.Path("/data/raw/jsons")

# Listar archivos
files = fs.listStatus(path)

# Mostrar nombres
for f in files:
    print(f.getPath().getName())



transaccion1.json
transaccion2.json
transaccion3.json
transaccion4.json
transaccion5(copia1).json
transaccion6.json
transacciones_generadas.json


In [37]:
# Leer entonces los archivos JSONs desde HDFS
df_json = (
    spark.read
    .option("multiLine", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .json("/data/raw/jsons/*.json")
)
df_json.printSchema()

# Vemos la estructura que detecto spark
df_json.limit(5).show(truncate=False)



root
 |-- amount: string (nullable = true)
 |-- authentication: struct (nullable = true)
 |    |-- failed_attempts: long (nullable = true)
 |    |-- method: string (nullable = true)
 |-- cardholder: struct (nullable = true)
 |    |-- age: string (nullable = true)
 |    |-- country: string (nullable = true)
 |-- device: struct (nullable = true)
 |    |-- device_trust_score: string (nullable = true)
 |    |-- device_type: string (nullable = true)
 |    |-- operating_system: string (nullable = true)
 |-- labels: struct (nullable = true)
 |    |-- is_fraud: boolean (nullable = true)
 |-- location: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- is_foreign_transaction: string (nullable = true)
 |    |-- location_mismatch: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- network_features: struct (nullable = true)
 |    |-- ip_risk_score: double (nullable = true)
 |    |-- velocity_last_24h: 

Como podemos observar, los JSON cuentan 12 columnas, las cuales son:
- transaction_id: identificador único de la transacción.
- timestamp: fecha y hora de la transacción.
- transaction_hour: hora de la transacción (0–23).
- amount: monto de la transacción.
- merchant_category: categoría del comercio.
- cardholder: información del titular de la tarjeta (edad y país).
- device: información del dispositivo usado (tipo, sistema operativo, score de confianza).
- network_features: características de la red (velocidad de transacciones, riesgo de IP).
- location: ciudad y país de la transacción, indica si es extranjera.
- authentication: método de autenticación y cantidad de intentos fallidos.
- labels: variable objetivo indicando si es fraude (0=normal | 1=fraude).
- schema_version: versión del esquema del dataset (muchos valores nulos).

Por otro lado, podemos notar que varias columnas que deberían ser numéricas (transaction_hour, amount, device_trust_score, velocity_last_24h, ip_risk_score) se encuentran como texto u objetos

Antes de comenzar con el proceso de limpieza, se realiza una inspección inicial del dataset con el objetivo de comprender su estructura, verificar el formato de las columnas y detectar posibles inconsistencias visibles a simple vista

In [38]:
df_json.select(df_json.columns[:5]).show(10, truncate=False)

+------+--------------+----------+-------------------------+-------+
|amount|authentication|cardholder|device                   |labels |
+------+--------------+----------+-------------------------+-------+
|168.04|{1, biometric}|{42, AR}  |{34.0, mobile, ios}      |{true} |
|84.09 |{1, pin}      |{38.0, US}|{100.0, desktop, windows}|{false}|
|51.11 |{1, pin}      |{45, AR}  |{30.0, mobile, android}  |{false}|
|6.17  |{1, pin}      |{38, BR}  |{44.0, mobile, windows}  |{false}|
|1.02  |{1, biometric}|{50, US}  |{57.0, tablet, macos}    |{true} |
|213.12|{1, pin}      |{55, CL}  |{95.0, tablet, windows}  |{false}|
|196.16|{1, biometric}|{38, AR}  |{85.0, tablet, linux}    |{false}|
|1.28  |{2, pin}      |{63, AR}  |{25.0, desktop, macos}   |{true} |
|124.9 |{1, biometric}|{49, AR}  |{88.0, mobile, windows}  |{false}|
|110.62|{0, biometric}|{19, US}  |{27.0, tablet, macos}    |{false}|
+------+--------------+----------+-------------------------+-------+
only showing top 10 rows



In [39]:
#Listamos columnas
df_json.columns

['amount',
 'authentication',
 'cardholder',
 'device',
 'labels',
 'location',
 'merchant_category',
 'network_features',
 'schema_version',
 'timestamp',
 'transaction_hour',
 'transaction_id']

In [40]:
#Listamos cantidad de filas
df_json.count()

5006

### Analizamos si se cargaron bien todos los archivos

In [41]:
#Analizamos si leyo todos los json

#Archivo transaccion1.json tenia la transaccion con id=TX-2024-00098321
df_json.where(
    "transaction_id = 'TX-2024-00098321'"
).select(
    "transaction_id",
    "timestamp",
    "transaction_hour",
    "amount"
).show(truncate=False)
#Vamos a ver que hay dos porque la transaccion5(copia1).json es identica con mismo id

+----------------+--------------------+----------------+------+
|transaction_id  |timestamp           |transaction_hour|amount|
+----------------+--------------------+----------------+------+
|TX-2024-00098321|2024-05-12T14:32:10Z|14              |245.5 |
|TX-2024-00098321|2024-05-12T14:32:10Z|14              |245.5 |
+----------------+--------------------+----------------+------+



In [42]:
#Archivo transaccion2.json tenia la transaccion con id=TX-2024-00011123
df_json.where(
    "transaction_id = 'TX-2024-00011123'"
).select(
    "transaction_id",
    "timestamp",
    "transaction_hour",
    "amount"
).show(truncate=False)

+----------------+--------------------+----------------+------+
|transaction_id  |timestamp           |transaction_hour|amount|
+----------------+--------------------+----------------+------+
|TX-2024-00011123|2024-03-02T10:15:41Z|10              |54.3  |
+----------------+--------------------+----------------+------+



In [43]:
#Archivo transaccion3.json tenia la transaccion con id=TX-2024-00045877
df_json.where(
    "transaction_id = 'TX-2024-00045877'"
).select(
    "transaction_id",
    "timestamp",
    "transaction_hour",
    "amount"
).show(truncate=False)

+----------------+--------------------+----------------+-------+
|transaction_id  |timestamp           |transaction_hour|amount |
+----------------+--------------------+----------------+-------+
|TX-2024-00045877|2024-07-11T03:42:19Z|3               |1299.99|
+----------------+--------------------+----------------+-------+



In [44]:
#Archivo transaccion4.json tenia la transaccion con id=TX-2024-00077201
df_json.where(
    "transaction_id = 'TX-2024-00077201'"
).select(
    "transaction_id",
    "timestamp",
    "transaction_hour",
    "amount"
).show(truncate=False)

+----------------+--------------------+----------------+------+
|transaction_id  |timestamp           |transaction_hour|amount|
+----------------+--------------------+----------------+------+
|TX-2024-00077201|2024-09-23T22:58:07Z|22              |420.0 |
+----------------+--------------------+----------------+------+



In [45]:
#Probamos archivo transaccion6.json tenia una transaccion con id=null pero un amount reconocible
df_json.where(
    "amount = 999999999999.99"
).select(
    "transaction_id",
    "timestamp",
    "transaction_hour",
    "amount"
).show(truncate=False)

+--------------+---------+----------------+------------------+
|transaction_id|timestamp|transaction_hour|amount            |
+--------------+---------+----------------+------------------+
|NULL          |NULL     |NULL            |9.9999999999999E11|
+--------------+---------+----------------+------------------+



Hemos corroborado que todos los archivos se han leido correctamente, asi que sigamos la limpieza

### Sacamos las estructuras internas y aplanamos (flatten). Tambien casteamos

En esta etapa se construye un nuevo dataframe (df1) a partir del dataset original (df_raw), con el objetivo de:
- Eliminar filas completamente vacías.
- Desanidar columnas que contienen diccionarios.
- Convertir variables al tipo de dato correspondiente.

In [46]:
from pyspark.sql import functions as F

df1 = df_json.select(
    # transaction
    F.col("transaction_id"),
    F.col("timestamp"),
    F.col("amount").cast("double").alias("amount"),
    F.col("transaction_hour").cast("int").alias("transaction_hour"),

    # merchant
    F.col("merchant_category"),

    # cardholder
    F.col("cardholder.age").cast("int").alias("cardholder_age"),
    F.col("cardholder.country").alias("cardholder_country"),

    # device
    F.col("device.device_type").alias("device_type"),
    F.col("device.operating_system").alias("device_os"),
    F.col("device.device_trust_score").cast("double").alias("device_trust_score"),

    # network
    F.col("network_features.velocity_last_24h").cast("double").alias("velocity_last_24h"),
    F.col("network_features.ip_risk_score").cast("double").alias("ip_risk_score"),

    # location
    F.col("location.city").alias("city"),
    F.col("location.country").alias("country"),
        (
        F.when(
            F.trim(F.lower(F.col("location.is_foreign_transaction"))).isin("true", "1", "1.0", "yes"),
            True,
        )
        .when(
            F.trim(F.lower(F.col("location.is_foreign_transaction"))).isin("false", "0", "0.0", "no"),
            False,
        )
        .otherwise(None)
        .alias("is_foreign_transaction")
    ),
    F.col("location.location_mismatch").cast("boolean").alias("location_mismatch"),

    # auth
    F.col("authentication.method").alias("auth_method"),
    F.col("authentication.failed_attempts").cast("int").alias("failed_attempts"),

    # label
    F.col("labels.is_fraud").alias("is_fraud"),

    # Schema version
    F.col("schema_version")
)

In [47]:
df1.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_hour: integer (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- cardholder_age: integer (nullable = true)
 |-- cardholder_country: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- device_os: string (nullable = true)
 |-- device_trust_score: double (nullable = true)
 |-- velocity_last_24h: double (nullable = true)
 |-- ip_risk_score: double (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- is_foreign_transaction: boolean (nullable = true)
 |-- location_mismatch: boolean (nullable = true)
 |-- auth_method: string (nullable = true)
 |-- failed_attempts: integer (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- schema_version: double (nullable = true)



Podemos observar ahora si que ya no todas las variables son del tipo string sino que de su tipo correspondiente. Por ejemplo, amount, transaction_hour, device_trust_score, velocity_last_24h, failed_attempt ahora son variables numericas

### Analizamos la cantidad de nulos, columnas sobrantes, duplicados

In [48]:
#Cantidad de nulos por columnas
from pyspark.sql.functions import col, sum, lit

null_counts = df1.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df1.columns
])

df_nulls = null_counts.selectExpr(
    "stack({}, {}) as (column, null_count)".format(
        len(df1.columns),
        ", ".join([f"'{c}', `{c}`" for c in df1.columns])
    )
)

df_nulls.show(truncate=False)

# Como podemos observar schema_version es una columna con casi todos nulos

+----------------------+----------+
|column                |null_count|
+----------------------+----------+
|transaction_id        |1         |
|timestamp             |1         |
|amount                |76        |
|transaction_hour      |32        |
|merchant_category     |0         |
|cardholder_age        |23        |
|cardholder_country    |1         |
|device_type           |0         |
|device_os             |0         |
|device_trust_score    |18        |
|velocity_last_24h     |22        |
|ip_risk_score         |0         |
|city                  |0         |
|country               |0         |
|is_foreign_transaction|11        |
|location_mismatch     |142       |
|auth_method           |0         |
|failed_attempts       |0         |
|is_fraud              |0         |
|schema_version        |4906      |
+----------------------+----------+



In [49]:
# Borramos schema_version y creamos un nuevo dataframe que sea df2
df2 = df1.drop("schema_version")
df2.columns

['transaction_id',
 'timestamp',
 'amount',
 'transaction_hour',
 'merchant_category',
 'cardholder_age',
 'cardholder_country',
 'device_type',
 'device_os',
 'device_trust_score',
 'velocity_last_24h',
 'ip_risk_score',
 'city',
 'country',
 'is_foreign_transaction',
 'location_mismatch',
 'auth_method',
 'failed_attempts',
 'is_fraud']

Del print anterior tambien podemos ver que hay un unico nulo en la columna de transaction_id asi que procederemos a borrar esa fila

In [50]:
#La columna transaction_id tiene una fila con nulo. Esta es la transaccion6.json. Borramos esa fila
df2 = df2.na.drop(subset=["transaction_id"])
#Lo mostramos de forma prolija
df_nulls = df2.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df2.columns
]).selectExpr(
    "stack({}, {}) as (column, null_count)".format(
        len(df2.columns),
        ", ".join([f"'{c}', `{c}`" for c in df2.columns])
    )
)

df_nulls.show(truncate=False)

+----------------------+----------+
|column                |null_count|
+----------------------+----------+
|transaction_id        |0         |
|timestamp             |0         |
|amount                |76        |
|transaction_hour      |31        |
|merchant_category     |0         |
|cardholder_age        |22        |
|cardholder_country    |0         |
|device_type           |0         |
|device_os             |0         |
|device_trust_score    |18        |
|velocity_last_24h     |22        |
|ip_risk_score         |0         |
|city                  |0         |
|country               |0         |
|is_foreign_transaction|11        |
|location_mismatch     |141       |
|auth_method           |0         |
|failed_attempts       |0         |
|is_fraud              |0         |
+----------------------+----------+



In [51]:
#La columna amount tampoco tiene que tener nulos. En vez de borrar, vamos a reemplazar por la media
from pyspark.sql.functions import avg, sum

#Calculamos la media
media_amount = (
    df2
    .select(avg(col("amount")).alias("media"))
    .first()["media"]
)
#Rellenamos los nulos con esa media
df2 = df2.na.fill(
    {"amount": media_amount}
)

#Lo mostramos de forma prolija
df_nulls = df2.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df2.columns
]).selectExpr(
    "stack({}, {}) as (column, null_count)".format(
        len(df2.columns),
        ", ".join([f"'{c}', `{c}`" for c in df2.columns])
    )
)

df_nulls.show(truncate=False)

+----------------------+----------+
|column                |null_count|
+----------------------+----------+
|transaction_id        |0         |
|timestamp             |0         |
|amount                |0         |
|transaction_hour      |31        |
|merchant_category     |0         |
|cardholder_age        |22        |
|cardholder_country    |0         |
|device_type           |0         |
|device_os             |0         |
|device_trust_score    |18        |
|velocity_last_24h     |22        |
|ip_risk_score         |0         |
|city                  |0         |
|country               |0         |
|is_foreign_transaction|11        |
|location_mismatch     |141       |
|auth_method           |0         |
|failed_attempts       |0         |
|is_fraud              |0         |
+----------------------+----------+



In [52]:
#Verificamos si hay filas duplicadas por id de transaccion
aux = (
    df2.groupBy("transaction_id")
       .count()
       .filter("count > 1")
)
#Mostramos ids duplicados
aux.show()
print(aux.count())


+----------------+-----+
|  transaction_id|count|
+----------------+-----+
|TX-2024-00003276|    2|
|TX-2024-00000319|    2|
|TX-2024-00001017|    2|
|TX-2024-00001493|    2|
|TX-2024-00002316|    2|
|TX-2024-00003199|    2|
|TX-2024-00000067|    2|
|TX-2024-00001221|    2|
|TX-2024-00000330|    2|
|TX-2024-00001042|    2|
|TX-2024-00002237|    2|
|TX-2024-00003552|    2|
|TX-2024-00000281|    2|
|TX-2024-00001019|    2|
|TX-2024-00000162|    2|
|TX-2024-00000644|    2|
|TX-2024-00000796|    2|
|TX-2024-00004120|    2|
|TX-2024-00000024|    2|
|TX-2024-00000049|    2|
+----------------+-----+
only showing top 20 rows

127


In [53]:
#Eliminamos duplicados por id de transaccion
df2 = df2.dropDuplicates(["transaction_id"])
#Verificamos si hay filas duplicadas
aux = (
    df2.groupBy("transaction_id")
       .count()
       .filter("count > 1")
)
#Cantidad de ids duplicados
print(aux.count())

0


In [54]:
from pyspark.sql.functions import col, sum
from pyspark.sql import Row

# Calculamos los nulos por columna
null_counts = df2.select([
    sum(col(c).isNull().cast("int")).alias(c) 
    for c in df2.columns
])

# Convertimos a formato "columna - nulos"
null_counts_long = null_counts.first().asDict()
df_nulls = spark.createDataFrame(
    [Row(columna=k, nulos=v) for k, v in null_counts_long.items()]
)

# Mostramos la tabla ordenada por cantidad de nulos
df_nulls.orderBy(col("nulos").desc()).show(truncate=False)

+----------------------+-----+
|columna               |nulos|
+----------------------+-----+
|location_mismatch     |139  |
|transaction_hour      |31   |
|velocity_last_24h     |22   |
|cardholder_age        |22   |
|device_trust_score    |18   |
|is_foreign_transaction|10   |
|cardholder_country    |0    |
|auth_method           |0    |
|device_os             |0    |
|device_type           |0    |
|timestamp             |0    |
|merchant_category     |0    |
|country               |0    |
|failed_attempts       |0    |
|ip_risk_score         |0    |
|is_fraud              |0    |
|amount                |0    |
|city                  |0    |
|transaction_id        |0    |
+----------------------+-----+



In [55]:
# Lista de columnas con nulos 
cols_to_clean = [
    "location_mismatch",
    "transaction_hour",
    "velocity_last_24h",
    "cardholder_age",
    "device_trust_score",
    "is_foreign_transaction"
]

# Eliminamos las filas con nulos en esas columnas
df2 = df2.dropna(subset=cols_to_clean)

In [56]:
from pyspark.sql.functions import col, sum
from pyspark.sql import Row

# Calculamos los nulos por columna
null_counts = df2.select([
    sum(col(c).isNull().cast("int")).alias(c) 
    for c in df2.columns
])

# Convertimos a formato "columna - nulos"
null_counts_long = null_counts.first().asDict()
df_nulls = spark.createDataFrame(
    [Row(columna=k, nulos=v) for k, v in null_counts_long.items()]
)

# Mostramos la tabla ordenada por cantidad de nulos
df_nulls.orderBy(col("nulos").desc()).show(truncate=False)

+----------------------+-----+
|columna               |nulos|
+----------------------+-----+
|amount                |0    |
|cardholder_age        |0    |
|failed_attempts       |0    |
|cardholder_country    |0    |
|is_foreign_transaction|0    |
|is_fraud              |0    |
|auth_method           |0    |
|transaction_id        |0    |
|merchant_category     |0    |
|device_trust_score    |0    |
|transaction_hour      |0    |
|location_mismatch     |0    |
|device_os             |0    |
|country               |0    |
|timestamp             |0    |
|velocity_last_24h     |0    |
|ip_risk_score         |0    |
|device_type           |0    |
|city                  |0    |
+----------------------+-----+



### Validamos rangos de los distintas columnas

En esta etapa se analizan los valores mínimos y máximos de las columnas numéricas del dataset. El objetivo es:
- Detectar valores fuera de rango (por ejemplo, horas mayores a 23).
- Identificar montos negativos o inconsistentes.
- Verificar que las variables cuantitativas respeten los límites esperados.

In [57]:
#Calculamos los rangos de valores de cada columna
from pyspark.sql import functions as F

num_cols = [c for c, t in df2.dtypes if t in ("int", "double")]

df_agg = df2.agg(
    *[F.min(c).cast("double").alias(f"{c}_min") for c in num_cols],
    *[F.max(c).cast("double").alias(f"{c}_max") for c in num_cols]
)
expr = ", ".join(
    [f"'{c}', {c}_min, {c}_max" for c in num_cols]
)

df_rangos = df_agg.selectExpr(
    f"stack({len(num_cols)}, {expr}) as (column, min, max)"
)

df_rangos.show(truncate=False)


+------------------+----+------------------+
|column            |min |max               |
+------------------+----+------------------+
|amount            |0.02|NaN               |
|transaction_hour  |0.0 |578.0             |
|cardholder_age    |18.0|2168.0            |
|device_trust_score|18.0|3091.5985282093216|
|velocity_last_24h |0.0 |100.66462167689161|
|ip_risk_score     |0.0 |0.74              |
|failed_attempts   |0.0 |3.0               |
+------------------+----+------------------+



Como podemos observar, hay varias inconsistencias. 

Por ejemplo, transaction-hour que deberia tener un rango de 0 a 23, tiene un calor maximo de 578. Lo mismo con cardholder con edades superiores a 2168

In [58]:
df2.count()

4662

In [59]:
# Filtramos las filas con rangos imposibles
df3 = df2.filter(
    (F.col("transaction_hour").between(0, 23)) &
    (F.col("cardholder_age").between(0, 120)) &
    (F.col("device_trust_score").between(0, 100)) &
    (F.col("velocity_last_24h").between(0, 10))
)
df3.count()

4573

In [60]:
#Calculamos los rangos de valores de cada columna
from pyspark.sql import functions as F

num_cols = [c for c, t in df2.dtypes if t in ("int", "double")]

df_agg = df3.agg(
    *[F.min(c).cast("double").alias(f"{c}_min") for c in num_cols],
    *[F.max(c).cast("double").alias(f"{c}_max") for c in num_cols]
)

expr = ", ".join(
    [f"'{c}', {c}_min, {c}_max" for c in num_cols]
)

df_rangos = df_agg.selectExpr(
    f"stack({len(num_cols)}, {expr}) as (column, min, max)"
)

df_rangos.show(truncate=False)

+------------------+----+-----+
|column            |min |max  |
+------------------+----+-----+
|amount            |0.02|NaN  |
|transaction_hour  |0.0 |23.0 |
|cardholder_age    |18.0|78.0 |
|device_trust_score|18.0|100.0|
|velocity_last_24h |0.0 |9.0  |
|ip_risk_score     |0.0 |0.74 |
|failed_attempts   |0.0 |3.0  |
+------------------+----+-----+



Ahora que hemos filtrado podemos observar que ya no hay valores atipicos para las variables antes mencionadas. 

### Transformamos columnas de categoricas a numericas (jsons)

In [61]:
df3.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- amount: double (nullable = false)
 |-- transaction_hour: integer (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- cardholder_age: integer (nullable = true)
 |-- cardholder_country: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- device_os: string (nullable = true)
 |-- device_trust_score: double (nullable = true)
 |-- velocity_last_24h: double (nullable = true)
 |-- ip_risk_score: double (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- is_foreign_transaction: boolean (nullable = true)
 |-- location_mismatch: boolean (nullable = true)
 |-- auth_method: string (nullable = true)
 |-- failed_attempts: integer (nullable = true)
 |-- is_fraud: boolean (nullable = true)



#### Voy a convertir unicamente las que despues voy usar en el csv

In [62]:
from pyspark.ml.feature import StringIndexer

# -------- transaction_hour --------
indexer_hour = StringIndexer(
    inputCol="transaction_hour",
    outputCol="transaction_hour_index",
    handleInvalid="keep"
)

df3 = indexer_hour.fit(df3).transform(df3)

df3 = df3.drop("transaction_hour")

df3 = df3.withColumnRenamed(
    "transaction_hour_index",
    "transaction_hour"
)


# -------- merchant_category --------
indexer_category = StringIndexer(
    inputCol="merchant_category",
    outputCol="merchant_category_index",
    handleInvalid="keep"
)

df3 = indexer_category.fit(df3).transform(df3)

df3 = df3.drop("merchant_category")

df3 = df3.withColumnRenamed(
    "merchant_category_index",
    "merchant_category"
)

df3.show(5)
df3.printSchema()


+----------------+--------------------+------+--------------+------------------+-----------+---------+------------------+-----------------+-------------+-------------+---------+----------------------+-----------------+-----------+---------------+--------+----------------+-----------------+
|  transaction_id|           timestamp|amount|cardholder_age|cardholder_country|device_type|device_os|device_trust_score|velocity_last_24h|ip_risk_score|         city|  country|is_foreign_transaction|location_mismatch|auth_method|failed_attempts|is_fraud|transaction_hour|merchant_category|
+----------------+--------------------+------+--------------+------------------+-----------+---------+------------------+-----------------+-------------+-------------+---------+----------------------+-----------------+-----------+---------------+--------+----------------+-----------------+
|TX-2024-00000001|2024-02-06T09:18:10Z|168.04|            42|                AR|     mobile|      ios|              34.0|      

In [63]:
df3.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- amount: double (nullable = false)
 |-- cardholder_age: integer (nullable = true)
 |-- cardholder_country: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- device_os: string (nullable = true)
 |-- device_trust_score: double (nullable = true)
 |-- velocity_last_24h: double (nullable = true)
 |-- ip_risk_score: double (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- is_foreign_transaction: boolean (nullable = true)
 |-- location_mismatch: boolean (nullable = true)
 |-- auth_method: string (nullable = true)
 |-- failed_attempts: integer (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- transaction_hour: double (nullable = false)
 |-- merchant_category: double (nullable = false)



## Limpieza de Logs (Datos no estructurados)


En esta etapa se trabaja con datos no estructurados provenientes de un archivo de logs del sistema.

A diferencia de los datasets estructurados analizados previamente (CSV y JSON), los logs no presentan un esquema tabular definido, sino que contienen información en formato texto plano.

El objetivo es leer el archivo línea por línea y convertirlo en un DataFrame para luego aplicar procesos de extracción y estructuración de la información relevante.
levante.

### Carga inicial de los archivos Logs

In [64]:
#Vemos que archivos hay en la carpeta /data/raw/logs
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(
    spark._jsc.hadoopConfiguration()
)

# Path a la carpeta
path = spark._jvm.org.apache.hadoop.fs.Path("/data/raw/logs")

# Listar archivos
files = fs.listStatus(path)

# Mostrar nombres
for f in files:
    print(f.getPath().getName())

aclaracionSupervisor.txt
logs_generados.log


In [65]:
# Como solo hay un archivo de logs y despues un archivo txt sobre una aclaracion del supervisor. Vamos a leerlos por separado

# Leer logs (formato JSON líneas)
df_logs = spark.read.text("/data/raw/logs/logs_generados.log")
df_logs.show(5, truncate=False)

# Leer aclaración del supervisor
df_aclaracion = spark.read.text("/data/raw/logs/aclaracionSupervisor.txt")
df_aclaracion.show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                                                                          |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|TransactionHour=10 TX-2024-27677439 User=5885 Amount=77480.47 ARS MerchantCategory=electronics Method=Transfer Statuss=APPROVED ForeignTransaction=1 LocationMismatch=0 DeviceTrustScore=51 VelocityLast24h=6.7 CardholderAge=70 

### Extracción de campos estructurados desde los logs

En esta etapa se comienzan a transformar los datos no estructurados en un formato analizable.

Para ello se utilizan expresiones regulares (regex), que permiten identificar patrones específicos dentro del texto y convertirlos en columnas estructuradas dentro del DataFrame.

In [66]:
from pyspark.sql.functions import regexp_extract
## Extraemos los campos fijos  (transaction_hour + transaction_id) desde los logs (df_logs tiene columna "value")
df_logs = (
    df_logs
    .withColumn("transaction_id", regexp_extract("value", r"(TX-\d{4}-\d+)", 1))
    .withColumn("transaction_hour", regexp_extract("value", r"TransactionHour=(\d+)", 1).cast("int"))  
)
df_logs.show(5, truncate=False)


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+----------------+
|value                                                                                                                                                                                                                                                          |transaction_id  |transaction_hour|
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+----------------+
|TransactionHour=10 TX-2024-27677439 User=5885 Amount=77480.47 ARS MerchantCategory=electronics Method=Transfer Statuss=APPR

In [67]:
#Bien ahora extraemos las otras columnas
from pyspark.sql.functions import regexp_extract, regexp_replace, lit

df_logs = (
    df_logs
    .withColumn("user_id", regexp_extract("value", r"User=(\d+)", 1))
    .withColumn("amount", regexp_replace(regexp_extract("value", r"Amount=([\d,\.]+)", 1), ",", ".").cast("double"))
    .withColumn("currency", regexp_extract("value", r"Amount=[\d,\.]+\s+([A-Z]{3})", 1))
    .withColumn("merchant_category", regexp_extract("value", r"MerchantCategory=(\w+)", 1))
    .withColumn("method", regexp_extract("value", r"Method=(\w+)", 1))
    .withColumn("status", regexp_extract("value", r"Status[s]?=(\w+)", 1))
    .withColumn("is_foreign_transaction", (regexp_extract("value", r"ForeignTransaction=(\d)", 1) == "1"))
    .withColumn("location_mismatch", (regexp_extract("value", r"LocationMismatch=(\d)", 1) == "1"))
    .withColumn("device_trust_score", regexp_extract("value", r"DeviceTrustScore=(\d+)", 1).cast("int"))
    .withColumn("velocity_last_24h", regexp_extract("value", r"VelocityLast24h=([\d\.]+)", 1).cast("double"))
    .withColumn("cardholder_age", regexp_extract("value", r"CardholderAge=(\d+)", 1).cast("int"))
    .withColumn("is_fraud", (regexp_extract("value", r"IsFraud=(true|false)", 1) == "true"))
    .withColumn("ip", regexp_extract("value", r"IP=([\d\.]+)", 1))
)

#Nos falta la columna ip_risk_score

### Analizamos como nos quedo el data frame

In [68]:
df_logs.printSchema()

root
 |-- value: string (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- transaction_hour: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- method: string (nullable = true)
 |-- status: string (nullable = true)
 |-- is_foreign_transaction: boolean (nullable = true)
 |-- location_mismatch: boolean (nullable = true)
 |-- device_trust_score: integer (nullable = true)
 |-- velocity_last_24h: double (nullable = true)
 |-- cardholder_age: integer (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- ip: string (nullable = true)



### Descripción de variables del dataset de logs

El dataset de logs contiene 23 columnas, resultantes de la transformación de texto no estructurado en un formato estructurado.

A continuación se describen las principales variables obtenidas:

- value: línea original del log en formato texto plano.
- transaction_id: identificador único de la transacción asociada al evento.
- transaction_hour: fecha y hora en la que se registró el evento en el log.
- user_id: identificador del usuario asociado a la transacción.
- amount: monto de la transacción.
- currency: moneda en la que se realizó la transacción.
- merchant_category: categoría del comercio.
- method: método utilizado en la transacción.
- status: estado del evento registrado (por ejemplo, aprobado o rechazado).
- is_foreign_transaction: indica si la transacción fue internacional (True/False).
- location_mismatch: indica si existe discrepancia entre la ubicación esperada y la detectada.
- device_trust_score: puntuación de confianza del dispositivo utilizado.
- velocity_last_24h: cantidad de transacciones realizadas en las últimas 24 horas.
- cardholder_age: edad del titular de la tarjeta.
- is_fraud: variable objetivo que indica si la transacción fue fraudulenta (True/False).
- failed_attempts: cantidad de intentos fallidos de autenticación.
- ip: dirección IP desde la cual se realizó la transacción.


In [69]:
#Vemos algunas columnas y algunas filas
df_logs.select(
    "transaction_id",
    "transaction_hour",
    "user_id",
    "amount",
    "currency",
    "status"
).show(10, truncate=False)


+----------------+----------------+-------+---------+--------+--------+
|transaction_id  |transaction_hour|user_id|amount   |currency|status  |
+----------------+----------------+-------+---------+--------+--------+
|TX-2024-27677439|10              |5885   |77480.47 |ARS     |APPROVED|
|TX-2024-43666932|11              |9499   |40368.68 |ARS     |FLAGGED |
|TX-2024-48265545|2               |208    |114638.99|ARS     |DECLINED|
|TX-2024-08606441|18              |987    |78865.06 |ARS     |FLAGGED |
|TX-2024-40141241|22              |293    |101118.72|ARS     |DECLINED|
|TX-2024-47112582|15              |3549   |62869.75 |ARS     |DECLINED|
|TX-2024-49317060|7               |7181   |128447.75|ARS     |FLAGGED |
|TX-2024-37724473|8               |9403   |7165.92  |ARS     |APPROVED|
|TX-2024-45773716|8               |424    |61770.02 |ARS     |FLAGGED |
|TX-2024-11245804|4               |5627   |63278.26 |ARS     |APPROVED|
+----------------+----------------+-------+---------+--------+--

In [70]:
#Vemos que tiene la colunma value
df_logs.select("value").show(1, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                                                                          |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|TransactionHour=10 TX-2024-27677439 User=5885 Amount=77480.47 ARS MerchantCategory=electronics Method=Transfer Statuss=APPROVED ForeignTransaction=1 LocationMismatch=0 DeviceTrustScore=51 VelocityLast24h=6.7 CardholderAge=70 

In [71]:
df_logs.count()

3000

In [72]:
#Vemos cuantos nulos tiene cada columna
df_nulls = df_logs.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_logs.columns
]).selectExpr(
    "stack({}, {}) as (column, null_count)".format(
        len(df_logs.columns),
        ", ".join([f"'{c}', `{c}`" for c in df_logs.columns])
    )
)

df_nulls.show(truncate=False)

+----------------------+----------+
|column                |null_count|
+----------------------+----------+
|value                 |0         |
|transaction_id        |0         |
|transaction_hour      |0         |
|user_id               |0         |
|amount                |0         |
|currency              |0         |
|merchant_category     |0         |
|method                |0         |
|status                |0         |
|is_foreign_transaction|0         |
|location_mismatch     |0         |
|device_trust_score    |0         |
|velocity_last_24h     |0         |
|cardholder_age        |0         |
|is_fraud              |0         |
|ip                    |0         |
+----------------------+----------+



### Transformamos columnas de categoricas a numericas (LOGS)

In [73]:
df_logs.printSchema()

root
 |-- value: string (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- transaction_hour: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- method: string (nullable = true)
 |-- status: string (nullable = true)
 |-- is_foreign_transaction: boolean (nullable = true)
 |-- location_mismatch: boolean (nullable = true)
 |-- device_trust_score: integer (nullable = true)
 |-- velocity_last_24h: double (nullable = true)
 |-- cardholder_age: integer (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- ip: string (nullable = true)



In [74]:
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(
    inputCol="merchant_category",
    outputCol="merchant_category_index",
    handleInvalid="keep"
)

df_logs = indexer.fit(df_logs).transform(df_logs)

# Eliminar la original
df_logs = df_logs.drop("merchant_category")

# Opcional: renombrar
df_logs = df_logs.withColumnRenamed(
    "merchant_category_index",
    "merchant_category"
)

df_logs.printSchema()


root
 |-- value: string (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- transaction_hour: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- method: string (nullable = true)
 |-- status: string (nullable = true)
 |-- is_foreign_transaction: boolean (nullable = true)
 |-- location_mismatch: boolean (nullable = true)
 |-- device_trust_score: integer (nullable = true)
 |-- velocity_last_24h: double (nullable = true)
 |-- cardholder_age: integer (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- ip: string (nullable = true)
 |-- merchant_category: double (nullable = false)



### Agregamos columna is_risk_store

La column `ip_risk_scoe` no se encuentra explícitamente en los archivos de logs, por lo que se construye de manera derivada utilizando las mismas reglas aplicadas previamente en los archivos JSON.

El objetivo es generarn **indicador numérico de riesgo entre 0 1**, combinando distintas señales de comportamiento y contexto de la transacción.

El cálculo parte de un valor ba de ***0.05**, al cual se le suman incrementos según determinadas condiciones:

- Se suma **0.25** si la transacción es internaional (`is_foreign_transactio = True`).
- Se suma **0.25** si existe discrepancia de ubcación (`location_mismath = True`).
- Se suma **0.15** si el monto eselevado (`amunt ≥ 500`).
- Se suma **0.10** si la velocidad transaccional en las últimas 24 hora es alta (`velocity_lat_24h ≥ 10`).
- Se suma **0.10** si el dispositivo presenta bajo nivel deconfianza (`device_trus_score ≤ 40`).

Estas condiciones se impleme"tan mediant" `numpy.where`, lo que permite realizar el cálculo de forma vectorizada sobre todo el DataFrame.

Finalmente, el resultado se norma"iza utiliza"do `.clip(0, 1)` para asegurar que el valor permanezca dentro del rango [0,1], se redondea a dos decimales y se convierte explíctamene a tipo `float`.

De esta manera, se obtiene una variable sintética que resume múltiples factores de riesgo en un único indicador cuantitativo, facilitando su utilización en etapas posteriores de análisis o modelado.

In [75]:
# ip_risk_score no esta en los logs pero lo calculamos a partir de los datos como en los archivos Json
from pyspark.sql import functions as F

df_logs2 = df_logs.withColumn(
    "ip_risk_score",
    F.round(
        F.greatest(
            F.lit(0.0),
            F.least(
                F.lit(1.0),
                F.lit(0.05)
                + F.when(F.col("is_foreign_transaction"), F.lit(0.25)).otherwise(F.lit(0))
                + F.when(F.col("location_mismatch") == 1, F.lit(0.25)).otherwise(F.lit(0))
                + F.when((F.col("amount").isNotNull()) & (F.col("amount") >= 500), F.lit(0.15)).otherwise(F.lit(0))
                + F.when((F.col("velocity_last_24h").isNotNull()) & (F.col("velocity_last_24h") >= 10), F.lit(0.10)).otherwise(F.lit(0))
                + F.when((F.col("device_trust_score").isNotNull()) & (F.col("device_trust_score") <= 40), F.lit(0.10)).otherwise(F.lit(0)),
            ),
        ),
        2,
    ).cast("double"),
)

In [76]:
#Vemos las columnas
df_logs2.columns

['value',
 'transaction_id',
 'transaction_hour',
 'user_id',
 'amount',
 'currency',
 'method',
 'status',
 'is_foreign_transaction',
 'location_mismatch',
 'device_trust_score',
 'velocity_last_24h',
 'cardholder_age',
 'is_fraud',
 'ip',
 'merchant_category',
 'ip_risk_score']

## Usamos aclaracion obtenida al principio

El log venia con un archivo de aclaraciones que decia lo siguiente: "Todos las transacciones que su id empiecen con "TX-2024-4..." tienen que considerarse como fraudes"

In [77]:
# Usamos la aclaracion para mejorar nuestros datos
df_aclaracion.show(truncate=False)

+--------------------------------------------------------------------------------------------------+
|value                                                                                             |
+--------------------------------------------------------------------------------------------------+
|Todos las transacciones que su id empiecen con "TX-2024-4..." tienen que considerarse como fraudes|
+--------------------------------------------------------------------------------------------------+



Contamos la cantidad de filas que cumplan esta condicion en los distintos dataframe

In [78]:
csv_count = df_credito.filter(col("transaction_id").startswith("TX-2024-4")).count()
csv_json = df3.filter(col("transaction_id").startswith("TX-2024-4")).count()
csv_logs = df_logs2.filter(col("transaction_id").startswith("TX-2024-4")).count()

print("Cantidad de filas que cumplen en csv:", csv_count)
print("Cantidad de filas que cumplen en json:", csv_json)
print("Cantidad de filas que cumplen en logs:", csv_logs)


Cantidad de filas que cumplen en csv: 0
Cantidad de filas que cumplen en json: 0
Cantidad de filas que cumplen en logs: 1000


Vemos la cantidad de transacciones fraudes en los logs

In [79]:
from pyspark.sql import functions as F

df_logs.groupBy("is_fraud").count().show()


+--------+-----+
|is_fraud|count|
+--------+-----+
|    true|  300|
|   false| 2700|
+--------+-----+



Cambiamos todas las tracciones que cumplen la condicion a fraudes

In [80]:
from pyspark.sql import functions as F
df_logs2 = df_logs.withColumn(
    "is_fraud",
    F.when(
        F.col("transaction_id").startswith("TX-2024-4"),
        F.lit(True)
    ).otherwise(F.col("is_fraud"))
)


Como se puede observar, el data frame quedo mas balanceado

In [81]:
from pyspark.sql import functions as F

df_logs2.groupBy("is_fraud").count().show()

+--------+-----+
|is_fraud|count|
+--------+-----+
|    true| 1195|
|   false| 1805|
+--------+-----+



### Tambien vamos a agregar una flag para las filas que cumplen esta condicion en todos los dataframes porque despues cuando querramos entrenar el arbol, no vamos a utilizar los ids de las transacciones 

In [82]:
from pyspark.ml.feature import VectorAssembler

# Crear columna binaria "id_flag"
df_logs2 = df_logs2.withColumn(
    "id_flag",
    when(col("transaction_id").startswith("TX-2024-4"), 1).otherwise(0)
)

df3 = df3.withColumn(
    "id_flag",
    when(col("transaction_id").startswith("TX-2024-4"), 1).otherwise(0)
)

df_credito = df_credito.withColumn(
    "id_flag",
    when(col("transaction_id").startswith("TX-2024-4"), 1).otherwise(0)
)



# Guardamos los archivos (En la carpeta Silver)


### Vamos a guardar los archivos en la carpeta silver que es capa de un data lake donde se guardan los datos ya transformados y limpios

El csv limpio y transformado lo guardamos en formato csv porque ya es un formato estructurado

In [83]:
df_credito.count()

8610

In [84]:
df_credito.write \
    .mode("overwrite") \
    .format("csv") \
    .save("/data/silver/csv_limpio")

Como los jsons tenian un formato semi-estructurado y los Logs un formato no estructurado, ahora que ya estan estructurados y limpios vamos a guardarlos
en formato parquet. Este es un formato de almacenamiento en columnas de código abierto diseñado para optimizar el procesamiento y análisis de grandes 
conjuntos de datos

In [85]:
df3.count()

4573

In [86]:
df_logs2.count()

3000

In [98]:
from pyspark.sql import functions as F

# Supongamos que tu DataFrame se llama df

# Para contar los nulos de todas las columnas:
df_logs2.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df_logs2.columns]).show()

+-----+--------------+----------------+-------+------+--------+------+------+----------------------+-----------------+------------------+-----------------+--------------+--------+---+-----------------+-------+
|value|transaction_id|transaction_hour|user_id|amount|currency|method|status|is_foreign_transaction|location_mismatch|device_trust_score|velocity_last_24h|cardholder_age|is_fraud| ip|merchant_category|id_flag|
+-----+--------------+----------------+-------+------+--------+------+------+----------------------+-----------------+------------------+-----------------+--------------+--------+---+-----------------+-------+
|    0|             0|               0|      0|     0|       0|     0|     0|                     0|                0|                 0|                0|             0|       0|  0|                0|      0|
+-----+--------------+----------------+-------+------+--------+------+------+----------------------+-----------------+------------------+-----------------+-----

In [87]:
df3.write \
    .mode("overwrite") \
    .format("parquet") \
    .save("/data/silver/jsons_limpios")

df_logs2.write \
    .mode("overwrite") \
    .format("parquet") \
    .save("/data/silver/logs_limpios")


# Guardamos la unificacion de los archivos (En la carpeta GOLD) 

### Se estandarizan los nombres de columnas, se aplican conversiones de tipo necesarias y se asegura la consistencia del esquema.

In [88]:
from pyspark.sql.functions import col, sum, when

df_logs2.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_logs2.columns
]).show()


+-----+--------------+----------------+-------+------+--------+------+------+----------------------+-----------------+------------------+-----------------+--------------+--------+---+-----------------+-------+
|value|transaction_id|transaction_hour|user_id|amount|currency|method|status|is_foreign_transaction|location_mismatch|device_trust_score|velocity_last_24h|cardholder_age|is_fraud| ip|merchant_category|id_flag|
+-----+--------------+----------------+-------+------+--------+------+------+----------------------+-----------------+------------------+-----------------+--------------+--------+---+-----------------+-------+
|    0|             0|               0|      0|     0|       0|     0|     0|                     0|                0|                 0|                0|             0|       0|  0|                0|      0|
+-----+--------------+----------------+-------+------+--------+------+------+----------------------+-----------------+------------------+-----------------+-----

In [89]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType

# --- CSV  ---
df_csv = df_credito.select(
    "transaction_id",
    "amount",
    "transaction_hour",
    "merchant_category",
    "foreign_transaction",
    "location_mismatch",
    "device_trust_score",
    "velocity_last_24h",
    "cardholder_age",
    "is_fraud",
    "id_flag"
)

# --- JSON---
df_json = df3.select(
    col("transaction_id").alias("transaction_id"),
    col("amount"),
    col("transaction_hour"),
    col("merchant_category"),
    col("is_foreign_transaction").cast(IntegerType()).alias("foreign_transaction"),
    col("location_mismatch").cast(IntegerType()).alias("location_mismatch"),
    col("device_trust_score"),
    col("velocity_last_24h"),
    col("cardholder_age"),
    col("is_fraud").cast(IntegerType()).alias("is_fraud"),
    col("id_flag")
)

# --- LOGS ---
df_logs = df_logs2.select(
    col("transaction_id").alias("transaction_id"),
    col("amount"),
    col("transaction_hour").cast(IntegerType()),
    col("merchant_category"),
    col("is_foreign_transaction").cast(IntegerType()).alias("foreign_transaction"),
    col("location_mismatch").cast(IntegerType()).alias("location_mismatch"),
    col("device_trust_score"),
    col("velocity_last_24h"),
    col("cardholder_age"),
    col("is_fraud").cast(IntegerType()).alias("is_fraud"),
    col("id_flag")
)

# --- UNION ALL ---
transactions_unified = df_csv.unionByName(df_json).unionByName(df_logs)

# Verificar 
transactions_unified.printSchema()

# Ver primeras filas
transactions_unified.show(5)


root
 |-- transaction_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_hour: double (nullable = true)
 |-- merchant_category: double (nullable = false)
 |-- foreign_transaction: integer (nullable = true)
 |-- location_mismatch: integer (nullable = true)
 |-- device_trust_score: double (nullable = true)
 |-- velocity_last_24h: double (nullable = true)
 |-- cardholder_age: integer (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- id_flag: integer (nullable = false)

+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--------+-------+
|transaction_id|amount|transaction_hour|merchant_category|foreign_transaction|location_mismatch|device_trust_score|velocity_last_24h|cardholder_age|is_fraud|id_flag|
+--------------+------+----------------+-----------------+-------------------+-----------------+------------------+-----------------+--------------+--

### No encontramos porque no termina de guardar en la carpeta Gold por lo tanto lo guardamos local

In [99]:
transactions_unified.write \
    .mode("overwrite") \
    .format("parquet") \
    .save("/data/gold/")

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [90]:
# Guardar gold en disco local del contenedor (evita fallos de HDFS)
# La ruta es la carpeta montada: queda en notebooks/data/gold en tu proyecto
import os
os.makedirs("/home/jovyan/work/data/gold", exist_ok=True)
transactions_unified.coalesce(1).write \
    .mode("overwrite") \
    .format("parquet") \
    .save("file:///home/jovyan/work/data/gold")